# Credit Risk & IFRS 9 Walkthrough

This notebook mirrors the user workflow of the repository:

1. generate a synthetic multi-period retail loan book,
2. fit a discrete-time survival PD model,
3. score one reporting date,
4. run the IFRS 9 ECL engine,
5. inspect monitoring and validation-style outputs.

References:
- Singer & Willett (1993): https://journals.sagepub.com/doi/10.3102/10769986018002155
- IFRS 9: https://www.ifrs.org/content/dam/ifrs/publications/pdf-standards/english/2021/issued/part-a/ifrs-9-financial-instruments.pdf
- SR 11-7: https://www.federalreserve.gov/boarddocs/srletters/2011/sr1107a1.pdf


In [11]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from credit_risk_lab import (
    PortfolioConfig,
    build_portfolio_report,
    fit_survival_pd_model,
    generate_portfolio_timeseries,
    run_ifrs9_pipeline,
    run_monitoring,
    score_portfolio,
)


In [12]:
dataset = generate_portfolio_timeseries(
    PortfolioConfig(random_seed=11, periods=12, num_borrowers=160, num_loans=280)
)
dataset.performance.head()


,loan_id,borrower_id,segment,region,snapshot_date,origination_date,term_quarters,quarters_on_book,remaining_term_quarters,balance,...,origination_pd_anchor,employment_stability,unemployment_sensitivity,unemployment_rate,policy_rate,house_price_index,house_price_growth,gdp_growth,default_next_period,prepayment_flag
0,L00008,B00057,mortgage,east,2022-03-31,2022-03-31,22,0,22,209501.48,...,0.072426,0.6009,1.3128,0.045051,0.023132,101.0899,0.0,0.010008,0,0
1,L00009,B00129,credit_card,west,2022-03-31,2022-03-31,6,0,6,8832.16,...,0.162465,0.5656,0.5027,0.045051,0.023132,101.0899,0.0,0.010008,0,0
2,L00017,B00015,auto,north,2022-03-31,2022-03-31,14,0,14,23617.00,...,0.149196,0.9800,0.9871,0.045051,0.023132,101.0899,0.0,0.010008,0,0
3,L00019,B00044,personal_loan,north,2022-03-31,2022-03-31,9,0,9,9068.66,...,0.042015,0.7428,0.9214,0.045051,0.023132,101.0899,0.0,0.010008,0,0
4,L00025,B00015,credit_card,north,2022-03-31,2022-03-31,12,0,12,15166.88,...,0.162465,0.9800,0.9871,0.045051,0.023132,101.0899,0.0,0.010008,0,0


In [13]:
model = fit_survival_pd_model(dataset.performance)
model.coefficient_table.head(10)


,feature,coefficient,std_error,z_score
0,const,-7.616931,1.592434,-4.783201
1,days_past_due,0.037295,0.008193,4.552159
2,dti,0.701176,1.272972,0.550818
3,ltv,1.273686,1.615832,0.788254
4,quarters_on_book,-0.113782,0.088204,-1.289989
5,rating_rank,0.636453,0.196688,3.235859
6,remaining_term_quarters,-0.005594,0.056756,-0.098555
7,segment_credit_card,-0.201097,0.751349,-0.267648
8,segment_mortgage,0.547516,0.782432,0.699762
9,segment_personal_loan,-0.027357,0.763797,-0.035817


In [14]:
as_of_date = dataset.performance['snapshot_date'].max()
scores = score_portfolio(dataset, model, as_of_date)
ecl_report = run_ifrs9_pipeline(dataset, scores)
monitoring = run_monitoring(
    reference_snapshot=scores.previous_snapshot_scores,
    current_snapshot=scores.snapshot_scores,
    reference_scores=scores.previous_snapshot_scores,
    current_scores=scores.snapshot_scores,
)
ecl_report.stage_summary


,stage,loan_count,exposure,mean_pd_12m,mean_lgd,allowance
0,1,122,10001596.81,0.014824,0.478476,30616.12
1,2,28,3030839.09,0.192375,0.355220,181313.19
2,3,6,631623.08,0.758116,0.740408,169522.74


In [15]:
print(build_portfolio_report(dataset, scores, ecl_report, monitoring))


# Credit Risk & IFRS 9 Summary

Reporting date: 2024-12-31

## Portfolio

- active loans: 156
- total balance: 13664058.98
- average DPD: 14.26
- average DTI: 0.4026

## Stage Summary

```text
 stage  loan_count    exposure  mean_pd_12m  mean_lgd  allowance
     1         122 10001596.81     0.014824  0.478476   30616.12
     2          28  3030839.09     0.192375  0.355220  181313.19
     3           6   631623.08     0.758116  0.740408  169522.74
```

## Scenario Summary

```text
scenario  weight  total_ecl
baseline     0.6  213759.94
downside     0.3  134681.35
  upside     0.1   33010.85
```

## Provision Roll-Forward

```text
                 component    amount
         opening_allowance 435228.16
          new_originations      0.00
maturities_and_prepayments   -988.50
                write_offs -33928.41
           stage_migration   3138.21
             remeasurement -21997.41
         closing_allowance 381452.05
```

## Score Drift

```text
      score      psi  wasserstein st